In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import zipfile

def gait_energy_image(sequence_images):
    """
    Compute Gait Energy Image.
    """
    gei = np.mean(sequence_images, axis=0)
    return (gei * 255).astype(np.uint8)

def create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=None):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    gei_images_folder = os.path.join(output_folder, 'gei_images')
    if not os.path.exists(gei_images_folder):
        os.makedirs(gei_images_folder)
    
    metadata = []
    counter = 0
    
    subject_folders = [f for f in os.listdir(base_folder) if f.isdigit()]
    if num_subjects is not None:
        subject_folders = subject_folders[:num_subjects]
    
    for subject_folder in subject_folders:
        counter += 1
        subject_path = os.path.join(base_folder, subject_folder, subject_folder)
        if not os.path.isdir(subject_path):
            print(f"No inner subject folder found for: {subject_folder}")
            continue
        
        for nm_folder in ['nm-01', 'nm-02', 'nm-03', 'nm-04', 'nm-05', 'nm-06']:
            nm_folder_path = os.path.join(subject_path, nm_folder)
            if not os.path.exists(nm_folder_path):
                print(f"No {nm_folder} folder found for subject: {subject_folder}")
                continue

            for cam_folder in camera_angles:
                cam_path = os.path.join(nm_folder_path, cam_folder)
                if not os.path.isdir(cam_path):
                    continue
                
                image_files = sorted([f for f in os.listdir(cam_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                
                if not image_files:
                    print(f"No images found in {cam_path}")
                    continue
                
                sequence_images = []
                for file in image_files:
                    image_path = os.path.join(cam_path, file)
                    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                    if image is not None and image.size > 0:
                        sequence_images.append(image)
                    else:
                        print(f"Failed to load image or empty image: {image_path}")
                
                if sequence_images:
                    gei = gait_energy_image(sequence_images)
                    
                    if gei is not None and gei.size > 0:
                        gei_filename = f"{subject_folder}_{nm_folder}_{cam_folder}.png"
                        gei_path = os.path.join(gei_images_folder, gei_filename)
                        cv2.imwrite(gei_path, gei)
                        
                        metadata.append({
                            'image_filename': gei_filename,
                            'label': int(subject_folder),
                            'cam_angle': int(cam_folder),
                            'nm_sequence': nm_folder
                        })
                    else:
                        print(f"Failed to create valid GEI for {cam_path}")
                
        print(f"{counter}: Finished processing subject: {subject_folder}")
    
    if metadata:
        df = pd.DataFrame(metadata)
        csv_path = os.path.join(output_folder, 'gei_metadata.csv')
        df.to_csv(csv_path, index=False)
        print(f"Successfully created {len(df)} GEI images for {len(df['label'].unique())} subjects.")
        print(f"Metadata saved to: {csv_path}")
    else:
        print("No GEI images were successfully created.")

def zip_dataset(output_folder, zip_filename):
    print(f"Creating zip file: {zip_filename}")
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_folder):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, output_folder)
                zipf.write(file_path, arcname)
    print(f"Zip file created: {zip_filename}")

# Example usage
if __name__ == "__main__":
    base_folder = '/kaggle/input/casia-b/GaitDatasetB-silh'
    output_folder = '/kaggle/working/gei_dataset'
    zip_filename = '/kaggle/working/gei_processed_dataset.zip'
    camera_angles = ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180']

    create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=camera_angles)
    zip_dataset(output_folder, zip_filename)
    print(f"GEI dataset has been created and zipped. You can now download {zip_filename} from the Kaggle output.")